# 文本预处理
:label:`sec_text_preprocessing`

对于序列数据处理问题，我们在 :numref:`sec_sequence`中
评估了所需的统计工具和预测时面临的挑战。
这样的数据存在许多种形式，文本是最常见例子之一。
例如，一篇文章可以被简单地看作一串单词序列，甚至是一串字符序列。
本节中，我们将解析文本的常见预处理步骤。
这些步骤通常包括：

1. 将文本作为字符串加载到内存中。
1. 将字符串拆分为词元（如单词和字符）。
1. 建立一个词表，将拆分的词元映射到数字索引。
1. 将文本转换为数字索引序列，方便模型操作。


In [1]:
import collections
import re
from d2l import torch as d2l

## 读取数据集

首先，我们从H.G.Well的[时光机器](https://www.gutenberg.org/ebooks/35)中加载文本。
这是一个相当小的语料库，只有30000多个单词，但足够我们小试牛刀，
而现实中的文档集合可能会包含数十亿个单词。
下面的函数(**将数据集读取到由多条文本行组成的列表中**)，其中每条文本行都是一个字符串。
为简单起见，我们在这里忽略了标点符号和字母大写。


In [3]:
#@save
# 向 d2l 的全局下载注册表 DATA_HUB 中登记一个数据集，键名为 'time_machine'
# 值的元组形式为 (下载地址, SHA-1 校验和)
#   - 下载地址：由 d2l.DATA_URL 前缀 + 文件名 'timemachine.txt' 拼接而成
#   - SHA-1 ：用于校验文件完整性，确保下载的内容未被损坏或篡改
d2l.DATA_HUB['time_machine'] = (d2l.DATA_URL + 'timemachine.txt',
                                '090b5e7e70c295757f55df93cb0a180b9691891a')

def read_time_machine():  #@save
    """将时间机器数据集加载到文本行的列表中"""
    # d2l.download 会先检查本地缓存：
    #   - 若已缓存且校验通过，直接返回本地文件路径；
    #   - 否则从网络下载到本地缓存后再返回路径。
    # 随后以只读文本模式 ('r') 打开该文件，with 语句保证用完后自动关闭文件句柄。
    with open(d2l.download('time_machine'), 'r') as f:
        # readlines() 按行读取，返回一个「字符串列表」，每个元素是一行文本
        # （每行末尾会带上换行符 '\n'）
        lines = f.readlines()
    # 使用列表推导式逐行清洗，返回清洗后的字符串列表：
    #   1) re.sub('[^A-Za-z]+', ' ', line)：正则匹配「一个或多个非字母字符」
    #      （数字、标点、空白等都算），把它们统一替换成「单个空格」，
    #      例如 "Hello, world!!" -> "Hello world "
    #   2) .strip()：去掉首尾的空白字符（空格、换行符等）
    #   3) .lower()：全部转为小写，使 "The" 与 "the" 视为同一个词元
    # 结果：每行只保留小写字母和单个空格，简化后续的词元化和词表构建
    return [re.sub('[^A-Za-z]+', ' ', line).strip().lower() for line in lines]

# 调用上面的函数，得到清洗后的文本行列表
lines = read_time_machine()
# 打印文本的总行数，用来确认数据规模
print(f'# 文本总行数: {len(lines)}')
# 打印第 1 行内容，观察清洗后的实际样式
print(lines[0])
# 打印第 11 行内容，进一步确认清洗效果
print(lines[10])

# 文本总行数: 3221
the time machine by h g wells
twinkled and his usually pale face was flushed and animated the


### `read_time_machine` 函数详解

一句话总览：

> 这个函数做的事：**下载《时光机器》文本 → 读成按行分开的列表 → 把每一行“洗干净”（只留小写字母和空格）**。

#### 1. 注册数据集（还不会下载）

```python
d2l.DATA_HUB['time_machine'] = (d2l.DATA_URL + 'timemachine.txt',
                                '090b5e7e70c295757f55df93cb0a180b9691891a')
```

这一步只是**登记信息**，相当于告诉 d2l 三件事：

- 名字叫 `time_machine`
- 从哪里下载（URL = `d2l.DATA_URL` 前缀 + 文件名）
- 下载后用什么校验完整性（SHA-1 校验和）

登记本身不会触发任何网络请求。

#### 2. 下载并打开文件

```python
with open(d2l.download('time_machine'), 'r') as f:
    lines = f.readlines()
```

- `d2l.download('time_machine')`：先查本地缓存，命中就返回本地路径，否则下载后再返回路径。
- `open(路径, 'r')`：以“只读文本”模式打开文件。
- `with ... as f`：`with` 语句保证用完自动关闭文件句柄，无需手动调用 `f.close()`。
- `f.readlines()`：把文件**按行读成一个列表**，每个元素是一整行字符串，且**末尾带着换行符 `\n`**。例如：

```python
lines = [
    "The Time Machine, by H. G. Wells!\n",
    "Chapter I\n",
    "\n",
    ...
]
```

#### 3. 逐行清洗（核心）

```python
return [re.sub('[^A-Za-z]+', ' ', line).strip().lower() for line in lines]
```

这行等价于下面这个循环，先看循环版本更容易理解：

```python
result = []
for line in lines:                              # 遍历每一行
    cleaned = re.sub('[^A-Za-z]+', ' ', line)   # ① 把非字母替换成空格
    cleaned = cleaned.strip()                   # ② 去掉首尾空白
    cleaned = cleaned.lower()                   # ③ 全部转为小写
    result.append(cleaned)                      # 加入新列表
return result
```

`[表达式 for line in lines]` 这种写法叫**列表推导式**，就是把上面的循环压缩成一行。它**不修改**原来的 `lines`，而是返回一个**新列表**。

#### 4. 正则 `re.sub('[^A-Za-z]+', ' ', line)` 拆解

`re.sub` 是 “regular expression substitute”（正则替换），固定吃三个参数：

```python
re.sub(模式, 要替换成什么, 在哪个字符串里找)
#      ↑          ↑            ↑
# '[^A-Za-z]+'  ' '         line
```

它的行为是：在 `line` 里找出**所有**匹配“模式”的片段，把每一个都替换成一个空格，然后返回**新字符串**。

模式 `'[^A-Za-z]+'` 的含义：

| 部分 | 名字 | 含义 |
| --- | --- | --- |
| `[` `]` | 字符集合 | 表示“方括号里这些字符中的某一个” |
| `^` | 在方括号**里面**表示取反 | “不是后面这些东西” |
| `A-Z` | 大写字母 | A 到 Z |
| `a-z` | 小写字母 | a 到 z |
| `+` | 量词 | 前面那个整体重复 **1 次或更多次** |

所以 `[^A-Za-z]+` = **一个或多个连续的、不是字母的字符**。所谓“非字母”包括：空格、逗号、句号、感叹号、数字、换行符 `\n`、下划线等——**除了 26 个大写和 26 个小写字母之外的一切**。

> ⚠️ `^` 有两副面孔：在 `[ ]` **外面**表示“字符串开头”；在 `[ ]` **里面**表示“取反”。这里在括号内，所以是取反。

`+` 的作用很关键：它让连续的非字母被当成**一整段**来匹配，于是 `"!!!"` 只会变成一个空格，而不是三个空格，也就顺便去掉了多余空白。

#### 5. 逐字符追踪一个例子

假设 `line = "The Time Machine, by H.G. Wells!\n"`（`\n` 是行末换行符，也是非字母）：

| 片段 | 类型 | 匹配 `[^A-Za-z]+`？ | 替换为 |
| --- | --- | --- | --- |
| `The` | 字母 | 否 | — |
| `" "` | 空格 | 是 | `" "` |
| `Time` | 字母 | 否 | — |
| `" "` | 空格 | 是 | `" "` |
| `Machine` | 字母 | 否 | — |
| `", "` | 逗号+空格 | 是 | `" "` |
| `by` | 字母 | 否 | — |
| `" "` | 空格 | 是 | `" "` |
| `H` | 字母 | 否 | — |
| `"."` | 句点 | 是 | `" "` |
| `G` | 字母 | 否 | — |
| `"."` | 句点 | 是 | `" "` |
| `" "` | 空格 | 是 | `" "` |
| `Wells` | 字母 | 否 | — |
| `"!\n"` | 感叹号+换行 | 是（一起匹配） | `" "` |

替换后得到 `"The Time Machine by H G Wells "`，再经过 `.strip()` 与 `.lower()`，最终变成：

```
"the time machine by h g wells"
```

#### 6. 两个必须知道的细节

1. **只认英文字母。** `[^A-Za-z]` 的字母集合只包含英文。若 `line` 中含有中文，中文字符也会被当成“非字母”而替换成空格（相当于删掉）。所以这个清洗逻辑是专门给英文语料用的。
2. **会“误伤”一些内容。** 例如 `"don't"` → `"don t"`，`"well-known"` → `"well known"`，`"3.14"` → `" "`。这是有意做的简化取舍：牺牲部分细节，换取干净、紧凑的词元。

#### 7. 最后三行的作用

```python
lines = read_time_machine()
print(f'# 文本总行数: {len(lines)}')
print(lines[0])
print(lines[10])
```

纯粹是打印出来验证效果：确认总行数、观察首行样式、再看第 11 行，检查清洗是否正确。

#### 8. 小结

- `DATA_HUB` 只是**登记**；`download` 才是**下载**。
- `readlines()` 得到**字符串列表**，每行一个元素（带 `\n`）。
- 列表推导式对**每一行**依次执行“替换非字母 → 去首尾空白 → 转小写”。
- `re.sub('[^A-Za-z]+', ' ', line)` 把标点、数字、多余空白都变成**单个空格**；这样后续 `tokenize` 用 `split()` 就能干净地分词，避免 `"machine,"` 与 `"machine"` 被当成两个不同词元。
- `#@save` 是 d2l 书籍的专用标记，表示“把这个函数保存进 d2l 包”，**不影响代码运行**。


## 词元化

下面的`tokenize`函数将文本行列表（`lines`）作为输入，
列表中的每个元素是一个文本序列（如一条文本行）。
[**每个文本序列又被拆分成一个词元列表**]，*词元*（token）是文本的基本单位。
最后，返回一个由词元列表组成的列表，其中的每个词元都是一个字符串（string）。


In [4]:
def tokenize(lines, token='word'):  #@save
    """将文本行拆分为单词或字符词元"""
    if token == 'word':
        return [line.split() for line in lines]
    elif token == 'char':
        return [list(line) for line in lines]
    else:
        print('错误：未知词元类型：' + token)

tokens = tokenize(lines)
for i in range(11):
    print(tokens[i])

['the', 'time', 'machine', 'by', 'h', 'g', 'wells']
[]
[]
[]
[]
['i']
[]
[]
['the', 'time', 'traveller', 'for', 'so', 'it', 'will', 'be', 'convenient', 'to', 'speak', 'of', 'him']
['was', 'expounding', 'a', 'recondite', 'matter', 'to', 'us', 'his', 'grey', 'eyes', 'shone', 'and']
['twinkled', 'and', 'his', 'usually', 'pale', 'face', 'was', 'flushed', 'and', 'animated', 'the']


## 词表

词元的类型是字符串，而模型需要的输入是数字，因此这种类型不方便模型使用。
现在，让我们[**构建一个字典，通常也叫做*词表*（vocabulary），
用来将字符串类型的词元映射到从$0$开始的数字索引中**]。
我们先将训练集中的所有文档合并在一起，对它们的唯一词元进行统计，
得到的统计结果称之为*语料*（corpus）。
然后根据每个唯一词元的出现频率，为其分配一个数字索引。
很少出现的词元通常被移除，这可以降低复杂性。
另外，语料库中不存在或已删除的任何词元都将映射到一个特定的未知词元“&lt;unk&gt;”。
我们可以选择增加一个列表，用于保存那些被保留的词元，
例如：填充词元（“&lt;pad&gt;”）；
序列开始词元（“&lt;bos&gt;”）；
序列结束词元（“&lt;eos&gt;”）。


In [4]:
class Vocab:  #@save
    """文本词表"""
    def __init__(self, tokens=None, min_freq=0, reserved_tokens=None):
        if tokens is None:
            tokens = []
        if reserved_tokens is None:
            reserved_tokens = []
        # 按出现频率排序
        counter = count_corpus(tokens)
        self._token_freqs = sorted(counter.items(), key=lambda x: x[1],
                                   reverse=True)
        # 未知词元的索引为0
        self.idx_to_token = ['<unk>'] + reserved_tokens
        self.token_to_idx = {token: idx
                             for idx, token in enumerate(self.idx_to_token)}
        for token, freq in self._token_freqs:
            if freq < min_freq:
                break
            if token not in self.token_to_idx:
                self.idx_to_token.append(token)
                self.token_to_idx[token] = len(self.idx_to_token) - 1

    def __len__(self):
        return len(self.idx_to_token)

    def __getitem__(self, tokens):
        if not isinstance(tokens, (list, tuple)):
            return self.token_to_idx.get(tokens, self.unk)
        return [self.__getitem__(token) for token in tokens]

    def to_tokens(self, indices):
        if not isinstance(indices, (list, tuple)):
            return self.idx_to_token[indices]
        return [self.idx_to_token[index] for index in indices]

    @property
    def unk(self):  # 未知词元的索引为0
        return 0

    @property
    def token_freqs(self):
        return self._token_freqs

def count_corpus(tokens):  #@save
    """统计词元的频率"""
    # 这里的tokens是1D列表或2D列表
    if len(tokens) == 0 or isinstance(tokens[0], list):
        # 将词元列表展平成一个列表
        tokens = [token for line in tokens for token in line]
    return collections.Counter(tokens)

我们首先使用时光机器数据集作为语料库来[**构建词表**]，然后打印前几个高频词元及其索引。


In [5]:
vocab = Vocab(tokens)
print(list(vocab.token_to_idx.items())[:10])

[('<unk>', 0), ('the', 1), ('i', 2), ('and', 3), ('of', 4), ('a', 5), ('to', 6), ('was', 7), ('in', 8), ('that', 9)]


现在，我们可以(**将每一条文本行转换成一个数字索引列表**)。


In [6]:
for i in [0, 10]:
    print('文本:', tokens[i])
    print('索引:', vocab[tokens[i]])

文本: ['the', 'time', 'machine', 'by', 'h', 'g', 'wells']
索引: [1, 19, 50, 40, 2183, 2184, 400]
文本: ['twinkled', 'and', 'his', 'usually', 'pale', 'face', 'was', 'flushed', 'and', 'animated', 'the']
索引: [2186, 3, 25, 1044, 362, 113, 7, 1421, 3, 1045, 1]


## 整合所有功能

在使用上述函数时，我们[**将所有功能打包到`load_corpus_time_machine`函数中**]，
该函数返回`corpus`（词元索引列表）和`vocab`（时光机器语料库的词表）。
我们在这里所做的改变是：

1. 为了简化后面章节中的训练，我们使用字符（而不是单词）实现文本词元化；
1. 时光机器数据集中的每个文本行不一定是一个句子或一个段落，还可能是一个单词，因此返回的`corpus`仅处理为单个列表，而不是使用多词元列表构成的一个列表。


In [7]:
def load_corpus_time_machine(max_tokens=-1):  #@save
    """返回时光机器数据集的词元索引列表和词表"""
    lines = read_time_machine()
    tokens = tokenize(lines, 'char')
    vocab = Vocab(tokens)
    # 因为时光机器数据集中的每个文本行不一定是一个句子或一个段落，
    # 所以将所有文本行展平到一个列表中
    corpus = [vocab[token] for line in tokens for token in line]
    if max_tokens > 0:
        corpus = corpus[:max_tokens]
    return corpus, vocab

corpus, vocab = load_corpus_time_machine()
len(corpus), len(vocab)

(170580, 28)

## 小结

* 文本是序列数据的一种最常见的形式之一。
* 为了对文本进行预处理，我们通常将文本拆分为词元，构建词表将词元字符串映射为数字索引，并将文本数据转换为词元索引以供模型操作。

## 练习

1. 词元化是一个关键的预处理步骤，它因语言而异。尝试找到另外三种常用的词元化文本的方法。
1. 在本节的实验中，将文本词元为单词和更改`Vocab`实例的`min_freq`参数。这对词表大小有何影响？


[Discussions](https://discuss.d2l.ai/t/2094)
